# Imports

In [2]:
import os
import time
import math
import json
import random
import pickle

from datetime import timedelta
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from tqdm import tqdm

from gensim.models import FastText

from sklearn.cluster import *
from sklearn.metrics.pairwise import cosine_similarity

from scipy.sparse.construct import random

from utils.model import VariationalAutoencoder
from utils.Loader import Train_Loader
from utils.tools import sanitize_string
from utils.tools import *

/home/cs/grad/nasrm1/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/usr/lib/python3/dist-packages/paramiko/pkey.py:82: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from cryptography.hazmat.primitives.ciphers.algorithms in 48.0.0.
  "cipher": algorithms.TripleDES,
/usr/lib/python3/dist-packages/paramiko/transport.py:261: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from cryptography.hazmat.primitives.ciphers.algorithms in 48.0.0.
  "class": algorithms.TripleDES,
/tmp/ipykernel_2209953/1337024824.py:23: DeprecationWarning: Please import `random` from the `scipy.sparse` namespace; the `scipy.sparse.construct` names

# Loading dataset

In [3]:
with open("dependancies/cadets_train.pkl", "rb") as file:
    training_df = pickle.load(file)

# Preparing filenames.txt
#### filenames.txt has the object files (destination node names) that FastText embedding model will be trained on later. 

In [ ]:
path_values = training_df['path']

with open('dependancies/filename_training.txt', 'w') as f:
    for path in path_values:
        if path !='':
            f.write(f"{path}\n")

print("File 'filename_training.txt' has been created with all paths.")

# Preparing cmdline.txt
#### cmdline.txt has the actor files (source node names) that FastText embedding model will be trained on later. 

In [ ]:
exec_values = training_df['exec']

with open('dependancies/cmdline_training.txt', 'w') as f:
    for exec in exec_values:
        if exec !='':
            f.write(f"{exec}\n")

print("File 'cmdline_training.txt' has been created with all paths.")

# Training FastText model on filenames

In [ ]:
epoch = 100
embedding_size = 256
input_file = 'dependancies/filename_training.txt'

print("Preprocessing")

pathname = []
corpus = []
f = open(input_file,'r')

while True:
    line = f.readline()
    if not line:
        break
    if line == '\n' or line == 'None' or line == 'none':
        continue
    splitline = sanitize_string(line)
    corpus.append(splitline)

print("Corpus size: ", len(corpus))

print("Building Vocab")
model = FastText(min_count = 5, vector_size=embedding_size, workers= 30, alpha=0.01,window=3,negative=5)
model.build_vocab(corpus)

print("Training")
model.train(corpus,epochs=epoch,total_examples=model.corpus_count)
model.save('dependancies/filepath-embedding_training.model')


# Training FastText model on cmdlines

In [ ]:
epoch = 100
embedding_size = 256
input_file = 'dependancies/cmdline_training.txt'
pathname = []
corpus = []
f = open(input_file,'r')

print("Preprocessing")

while True:
    line = f.readline()
    if not line:
        break
    if line == '\n' or line == 'None' or line == 'none':
        continue

    splitline = sanitize_string(line)
    corpus.append(splitline)

print("Corpus size: ", len(corpus))

print("Building Vocab")
model = FastText(min_count = 2, vector_size=embedding_size, workers= 30, alpha=0.01,window=3,negative=5)
model.build_vocab(corpus)

print("Training")
model.train(corpus,epochs=epoch,total_examples=model.corpus_count)
model.save('dependancies/cmdline-embedding_training.model')


# Preparing process-event-benign-training.txt
#### This code creates a file that has each process and the other files it interacted with in the training dataset, so that we train an autoencoder on reconstructing their embeddings.

In [ ]:
output_file = '/dependancies/process-event-benign-training.txt'

# Step 1: Build mappings
actor_info = {}  # actorID -> (exec, pid) [pid is not in your data, so we'll use actorID as a placeholder]
actor_to_files = defaultdict(set)  # actorID -> set of file paths

print("Processing DataFrame...")
for _, row in tqdm(training_df.iterrows(), total=len(training_df)):
    actor_id = row['actorID']
    exec_cmd = row['exec'].lower()  # Using 'exec' as the command line
    filepath = row['path'].lower()  # Using 'path' as the file path

    # Store actor info (exec and a placeholder PID)
    if actor_id not in actor_info:
        actor_info[actor_id] = (exec_cmd, str(actor_id))  

    # Track files accessed by this actor (exec)
    actor_to_files[actor_id].add(filepath)

# Step 2: Write to output file
print(f"Writing output to {output_file}...")
with open(output_file, 'w') as out:
    for actor_id, (exec_cmd, pid) in actor_info.items():
        if actor_id not in actor_to_files:
            continue  # Skip if no paths interacted with

        # Write process header line: exec$$$pid$$$false
        out.write(f"{exec_cmd}$$${pid}$$$false")

        # Write up to 6 associated paths
        file_list = list(actor_to_files[actor_id])
        for fpath in file_list:
            out.write(f"{fpath}\n")

        # Blank line separator
        out.write("\n")

print("Done.") 

# Calculating weights
#### It computes the statistical importance of different file paths and assesses the behavioral consistency of processes. This is achieved by implementing a custom TF-IDF (Term Frequency-Inverse Document Frequency) weighting scheme and a Stability metric using clustering.

In [ ]:
def extract_process_feature_benign(file_path, w2v):

    f = open(file_path, 'r')
    id = 0  # This is used as a unique process ID, incremented every time we hit a blank line (i.e., end of one process block)

    file_freq = defaultdict(set)  # Keeps track of which process IDs each file appears in
    file_vec = []  # List to store the embedding vector for each unique file path
    fileid2name = []  # List to store the canonical name (string) for each unique file path

    while True:
        line = f.readline()

        # If it's a blank line, increment the process ID
        if line == '\n':
            id += 1

        # End of file
        if not line:
            break

        # Clean the line (remove trailing newline, lowercase)
        filepath = line.strip().lower()

        # Skip lines that end with $$$true or $$$false (used for ground truth labeling elsewhere)
        if filepath.endswith('$$$true') or filepath.endswith('$$$false'):
            continue

        # Tokenize the file path into components using sanitize_string (e.g., split by slashes, remove punctuation)
        split_path = sanitize_string(filepath)

        # Create a normalized string version of the path for indexing
        newname = '/'.join(split_path)

        # Skip empty results (e.g., if path couldn't be tokenized)
        if len(split_path) == 0:
            continue

        # If this file has not been seen before, compute its embedding
        if newname not in fileid2name:
            tmp = []
            for token in split_path:
                tmp += [w2v.wv[token]]  # Get the FastText embedding for each token
            r = np.mean(tmp, axis=0).tolist()  # Take the mean of all token vectors to represent the full path
            file_vec.append(r)  # Save the embedding
            fileid2name.append(newname)  # Save the canonical name

        # Record that this file appears in this process ID
        file_freq[newname].add(id)

    return file_vec, fileid2name, file_freq, id


In [ ]:
def extract_process_vec(file_path, tfidf, w2v, c2v):
    process_map = {}  # Maps process ID to the command line string
    f = open(file_path, 'r')
    process_vec = defaultdict(list)  # Final output: maps process name to a list of vector representations
    id = 0  # Process ID counter
    
    # Compute the mean and max TF-IDF scores, used for normalization
    mean_s = np.mean(list(tfidf.values()))
    max_s = np.max(list(tfidf.values()))
    
    isprocess_file = True  # Flag indicating whether we're processing the first file (command) for this process
    tmp_process_vec = []  # Temporary list to hold the current process's feature vectors
    ground_truth = {}  # Maps process ID to its ground-truth label if present
    new_cmd = ''  # Holds the command line for the current process

    while True:
        line = f.readline()
        
        # End of current process block (blank line indicates separation)
        if line == '\n':
            if tmp_process_vec:
                # Take the mean vector for the process and store it
                process_vec[pname].append(np.mean(tmp_process_vec, axis=0).tolist())
            id += 1
            tmp_process_vec = []
            isprocess_file = True
            continue
        
        # End of file
        if not line:
            break

        # Clean and standardize path
        filepath = line.strip().lower()

        # Handle ground truth labels and strip them from filepath
        if filepath.endswith('$$$true'):
            filepath = filepath.replace('$$$true','')
            ground_truth[id] = filepath
        else:
            filepath = filepath.replace('$$$false','')

        # Extract command name if present
        if '$$$' in filepath:
            filepath, pname = filepath.split('$$$')[0], filepath.split('$$$')[1]

        # Tokenize the path (e.g., from path to words like ["windows", "system32", "svchost.exe"])
        split_path = sanitize_string(filepath)
        if len(split_path) == 0:
            continue

        # First file is the executable/command line
        if isprocess_file:
            new_cmd = '/'.join(split_path)
            process_map[id] = new_cmd  # Save the reconstructed command string
            isprocess_file = False

            # Get vector from command-line model (c2v)
            tmp = []
            for token in split_path:
                tmp += [c2v.wv[token]]
            r = np.mean(tmp, axis=0)

            # Scale by average TF-IDF to match file vector magnitudes
            r = r * mean_s

        else:
            # Other files accessed by the process
            tmp = []
            for token in split_path:
                tmp += [w2v.wv[token]]
            r = np.mean(tmp, axis=0)

            # Compute TF-IDF weight
            newname = '/'.join(split_path)
            s = tfidf.get(newname, mean_s)  # Use TF-IDF if present, else mean
            r = r * s

        # Add this file's (or command's) vector to the current process
        tmp_process_vec.append(r.tolist())

    return process_vec, process_map, ground_truth


In [ ]:
inputfile = 'dependancies/process-event-benign-training.txt'
w2v_dic = 'dependancies/filepath-embedding_training.model'
w2v = FastText.load(w2v_dic)
c2v_dic = 'dependancies/cmdline-embedding_training.model'
c2v = FastText.load(c2v_dic)

file_vec, fileid2name, file_freq, process_num = extract_process_feature_benign(inputfile, w2v)

threshold = 0.9
cos = cosine_similarity(np.array(file_vec))
tfidf_dic = {}
for i in range(cos.shape[0]):
    index = np.where(cos[i] > threshold)[0]  
    process_set = set()
    for j in index:
        process_set |= file_freq[fileid2name[j]] 
    process_set |= file_freq[fileid2name[i]]
    tfidf_dic[fileid2name[i]] = math.log(process_num / len(process_set), 2)

# Save the computed TF-IDF scores to a JSON file
json.dump(tfidf_dic, open('dependancies/tfidf.json', 'w'))

process_vec, process_map, ground_truth = extract_process_vec(inputfile, tfidf_dic, w2v, c2v)

stability = {}

for process in tqdm(process_vec.keys()):
    refer_words = process_vec[process]
    
    if len(np.array(refer_words).flatten()) == 0:
        continue
    
    if len(refer_words) > 20000:
        idx = np.random.choice(range(len(refer_words)), 20000)
        refer_words = np.array(refer_words)[idx]

    clustering = DBSCAN(eps=0.05, metric='cosine', min_samples=5).fit(refer_words)
    s = len(set(clustering.labels_.tolist()))
    stability[process] = s

f = open('dependancies/stability-embedding.json', 'w')

json.dump(stability, f)

f.close()

# Training a VAE on reconstructing benign embeddings
#### It involves building and optimizing a Variational Autoencoder (VAE) to learn the latent distribution of "normal" system behavior. The model is trained exclusively on process embeddings derived from the benign training dataset

In [ ]:
def extract_process_feature_anomaly(file_path,tfidf, stability, w2v, c2v):
    process_map = {}
    f = open(file_path,'r')
    process_vec = defaultdict(list)
    id = 0
    mean_s = np.mean(list(tfidf.values()))
    max_s = np.max(list(tfidf.values()))
    isprocess_file = True
    tmp_process_vec = []
    cmdline_vec = []
    ground_truth = {}
    while True:
        line = f.readline()
        if line == '\n':
            process_vec[id] = np.mean(tmp_process_vec,axis=0).tolist()

            id += 1
            tmp_process_vec = []
            cmdline_vec = []
            isprocess_file = True
            continue
        if not line:
            break

        filepath = line.strip().lower()
        if filepath.endswith('$$$true'):
            filepath = filepath.replace('$$$true','')
            ground_truth[id] = filepath
        else:
            filepath = filepath.replace('$$$false','')

        if '$$$' in filepath:
            filepath, pname = filepath.split('$$$')[0], filepath.split('$$$')[1]


        split_path = sanitize_string(filepath)
        if len(split_path) == 0:
            continue
        if isprocess_file:
            process_map[id] = pname
            isprocess_file = False
            tmp = []
            for l,i in enumerate(split_path):
                tmp += [c2v.wv[i]]
            r = np.mean(tmp,axis=0)
            r = r * mean_s
            
        else:
            tmp = []
            for l,i in enumerate(split_path):
                tmp += [w2v.wv[i]]
            r = np.mean(tmp,axis=0)
            newname = '/'.join(split_path)
            if newname in tfidf:
                s = tfidf[newname]
            else:
                s = mean_s
            r = r * s

        tmp_process_vec.append(r.tolist())
    return process_vec, process_map, ground_truth


In [ ]:
w2v_dic = 'dependancies/filepath-embedding_training.model'
w2v = FastText.load(w2v_dic)
c2v_dic = 'dependancies/cmdline-embedding_training.model'
c2v = FastText.load(c2v_dic)
benign_data_file = 'dependancies/process-event-benign-training.txt'
tfidf_file = 'dependancies/tfidf.json'
tfidf_dic = json.load(open(tfidf_file))
stability_file = 'dependancies/stability-embedding.json'
stability = json.load(open(stability_file))

In [ ]:
process_vec, process_map, x = extract_process_feature_anomaly(benign_data_file, tfidf_dic, stability, w2v,c2v)

In [ ]:

keys_to_remove = []

for k, v in process_vec.items():
    # Case 1: v is a single vector (list of floats or a float)
    if isinstance(v, float):
        if np.isnan(v):
            keys_to_remove.append(k)
    elif isinstance(v, list) and all(isinstance(i, float) for i in v):
        if np.isnan(v).any():
            keys_to_remove.append(k)
    # Case 2: v is a list of vectors
    elif isinstance(v, list):
        for vec in v:
            if isinstance(vec, list) and np.isnan(vec).any():
                keys_to_remove.append(k)
                break

# Remove keys
for k in keys_to_remove:
    del process_vec[k]

print('len process vec: ', len(process_vec))
print('len process map: ', len(process_map))

#### split the benign data into train data and valid data
data_len = len(list(process_vec.keys()))
print(data_len)
train_data = defaultdict(list)
train_data2 = defaultdict(list)
per = data_len
cnt = 0
keys = list(process_vec.keys())
random.shuffle(keys)

train_data = process_vec
    
print('train:', len(list(train_data.keys())))
out_embedd1 = 'dependancies/process_embedding_train.json'
out1 = open(out_embedd1,'w')
json.dump(train_data, out1)
out1.close()
train_data.clear()


####################
#VAE Train
####################

print('start to train VAE')

batch_size = 128
lr = 0.001         # learning rate
w_d = 1e-5        # weight decay
momentum = 0.9


train_file = 'dependancies/process_embedding_train.json'
train_set = Train_Loader(train_file)

train_ = torch.utils.data.DataLoader(
        train_set,
        batch_size=batch_size,
        shuffle=True,
        pin_memory=False,
    )

metrics = defaultdict(list)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = VariationalAutoencoder(32)
model.to(device)
criterion = nn.MSELoss(reduction='sum')
optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=w_d)
model.train()
min_epoch_loss = 100
start = time.time()

epochs = 1

for epoch in range(epochs):
    ep_start = time.time()
    running_loss = 0.0
    for bx, (data) in enumerate(train_):
        data = data.to(device)
        sample = model(data)
        loss = criterion(data.to(device), sample) + model.encoder.kl
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    epoch_loss = running_loss/len(train_set)
    metrics['train_loss'].append(epoch_loss)
    ep_end = time.time()
    print('-----------------------------------------------')
    print('[EPOCH] {}/{}\n[LOSS] {}'.format(epoch+1,epochs,epoch_loss))
    print('Epoch Complete in {}'.format(timedelta(seconds=ep_end-ep_start)))
    # if epoch_loss < min_epoch_loss:
        # torch.save(model,'./AE.model')
    torch.save(model.state_dict(), 'dependancies/AE.model')
    min_epoch_loss = epoch_loss

end = time.time()
print('-----------------------------------------------')
print('[System Complete: {}]'.format(timedelta(seconds=end-start)))

In [ ]:
w2v_dic = 'dependancies/filepath-embedding_training.model'
w2v = FastText.load(w2v_dic)
c2v_dic = 'dependancies/cmdline-embedding_training.model'
c2v = FastText.load(c2v_dic)
benign_data_file = 'dependancies/process-event-benign-training.txt'
tfidf_file = 'dependancies/tfidf.json'
tfidf_dic = json.load(open(tfidf_file))
stability_file = 'dependancies/stability-embedding.json'
stability = json.load(open(stability_file))

device = 'cuda' if torch.cuda.is_available() else 'cpu'

model = VariationalAutoencoder(32)
model.load_state_dict(torch.load('dependancies/AE.model'))
model.to(device)
model.eval()

criterion = nn.MSELoss(reduction='sum')

valid_data = json.load(open('dependancies/process_embedding_train.json'))

In [ ]:
label = []
process_name = []
loss_dist = []

model.eval()
for i in valid_data:
    try:
        name = process_map[i]
    except:
        name = 'None' 
    data = torch.FloatTensor(valid_data[i])
    sample = model(data.to(device))
    loss = criterion(data.to(device),sample).item()
    if name in stability:
        loss = loss / (math.log(stability[name]) + 1)
    loss_dist.append(loss)

anomaly_std = np.std(np.array(loss_dist))
anomaly_mean = np.mean(np.array(loss_dist))

anomaly_cutoff = np.percentile(np.array(loss_dist),90)

print(np.percentile(np.array(loss_dist),50))
print(np.percentile(np.array(loss_dist),60))
print(np.percentile(np.array(loss_dist),70))
print(np.percentile(np.array(loss_dist),80))
print(np.percentile(np.array(loss_dist),90))

print(anomaly_mean,anomaly_std)
print('anomaly threshold: ', anomaly_cutoff)